In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail")

bronze = spark.table("workspace.retail.bronze_online_retail")
bronze_count = bronze.count()
print("Bronze rows (must be fully accounted for):", bronze_count)
bronze.printSchema()

In [0]:
OVERLAP_START = "2010-12-01 00:00:00"

NON_PRODUCT_CODES = [
    "POST", "DOT", "M", "C2", "C3", "D", "S", "BANK CHARGES",
    "ADJUST", "ADJUST2", "AMAZONFEE", "CRUK", "TEST001", "TEST002",
    "B", "GIFT",
]
VOUCHER_PREFIX = "GIFT_0001"   # gift_0001_10, _20, ... (vouchers, not goods)

business_cols = ["invoice", "stock_code", "description", "quantity",
                 "invoice_date", "price", "customer_id", "country"]

In [0]:
std = (
    bronze
    .withColumn("invoice_clean", F.upper(F.trim(F.col("invoice"))))
    .withColumn("stock_code_clean", F.upper(F.trim(F.col("stock_code"))))
    .withColumn("description_clean", F.trim(F.col("description")))
    .withColumn("customer_id_clean", F.coalesce(F.col("customer_id").cast("long"), F.lit(-1)))
    .withColumn("is_cancellation", F.col("invoice_clean").startswith("C"))
)

In [0]:
def flag(cond):
    # a NULL in a comparison would silently count as "not matched"; force it to False
    return F.coalesce(cond, F.lit(False))

flagged = (
    std
    .withColumn("f_overlap",
        flag(F.col("source_file").contains("2009_2010")
             & (F.col("invoice_date") >= F.lit(OVERLAP_START).cast("timestamp"))))
    .withColumn("f_missing_required",
        F.col("invoice").isNull() | F.col("stock_code").isNull() | F.col("quantity").isNull()
        | F.col("price").isNull() | F.col("invoice_date").isNull())
    .withColumn("f_non_product",
        flag(F.col("stock_code_clean").isin(NON_PRODUCT_CODES)
             | F.col("stock_code_clean").startswith(VOUCHER_PREFIX)))
    .withColumn("f_cancel_pos_qty", flag(F.col("is_cancellation") & (F.col("quantity") > 0)))
    .withColumn("f_stock_adjustment", flag(~F.col("is_cancellation") & (F.col("quantity") < 0)))
    .withColumn("f_invalid_price", flag(F.col("price") <= 0))
)

In [0]:
flagged = flagged.withColumn(
    "reject_reason",
    F.when(F.col("f_overlap"), "OVERLAP_DUPLICATE")
     .when(F.col("f_missing_required"), "MISSING_REQUIRED_FIELD")
     .when(F.col("f_non_product"), "NON_PRODUCT_CODE")
     .when(F.col("f_cancel_pos_qty"), "CANCELLATION_POSITIVE_QTY")
     .when(F.col("f_stock_adjustment"), "STOCK_ADJUSTMENT")
     .when(F.col("f_invalid_price"), "INVALID_PRICE")
)

In [0]:
rejects = (
    flagged
    .filter(F.col("reject_reason").isNotNull())
    .select(*bronze.columns, "reject_reason")
    .withColumn("rejected_at", F.current_timestamp())
)

(rejects.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.retail.dq_rejects"))

In [0]:
clean = flagged.filter(F.col("reject_reason").isNull())

w = Window.partitionBy(*business_cols)

silver = (
    clean
    .withColumn("dup_group_size", F.count("*").over(w))
    .withColumn("dup_seq", F.row_number().over(w.orderBy("source_row_id")))
    .withColumn("is_possible_duplicate", F.col("dup_group_size") > 1)
    .withColumn("line_amount", F.round(F.col("quantity") * F.col("price"), 2))
    .withColumn("invoice_type",
        F.when(F.col("is_cancellation"), "CANCELLATION")
         .when(F.col("invoice_clean").startswith("A"), "ADJUSTMENT")
         .otherwise("SALE"))
    .withColumn("invoice_month", F.trunc(F.to_date("invoice_date"), "month"))
    .select(
        F.col("invoice_clean").alias("invoice"),
        "invoice_type", "is_cancellation",
        F.col("stock_code_clean").alias("stock_code"),
        F.col("description_clean").alias("description"),
        "quantity", "price", "line_amount",
        "invoice_date", "invoice_month",
        F.col("customer_id_clean").alias("customer_id"),
        (F.col("customer_id_clean") == -1).alias("is_unknown_customer"),
        "country",
        "dup_group_size", "dup_seq", "is_possible_duplicate",
        "source_file", "source_row_id", "ingested_at",
    )
)

(silver.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.retail.silver_online_retail"))

In [0]:
silver_t = spark.table("workspace.retail.silver_online_retail")
rej_t = spark.table("workspace.retail.dq_rejects")

s, r = silver_t.count(), rej_t.count()
print(f"Silver: {s:,} | Rejects: {r:,} | Sum: {s + r:,} | Bronze: {bronze_count:,}")
assert s + r == bronze_count, "Rows missing or double-counted!"

# 1. Rejects by reason, with the money value excluded
display(
    rej_t.groupBy("reject_reason")
    .agg(F.count("*").alias("rows"),
         F.round(F.sum(F.col("quantity") * F.col("price")), 2).alias("value_excluded"))
    .orderBy(F.desc("rows"))
)

# 2. Hard guarantees about the clean data
assert rej_t.filter("reject_reason = 'OVERLAP_DUPLICATE'").count() == 22523
assert silver_t.filter("price <= 0").count() == 0
assert silver_t.filter("quantity < 0 AND NOT is_cancellation").count() == 0
assert silver_t.filter(F.col("customer_id").isNull()).count() == 0
assert silver_t.filter(F.col("stock_code").isin(NON_PRODUCT_CODES)).count() == 0
print("All validation checks passed")

# 3. Revenue at risk from possible duplicates
display(
    silver_t.agg(
        F.sum(F.col("is_possible_duplicate").cast("int")).alias("rows_in_dup_groups"),
        F.sum((F.col("dup_seq") > 1).cast("int")).alias("extra_copies"),
        F.round(F.sum(F.when(F.col("dup_seq") > 1, F.col("line_amount"))), 2).alias("revenue_at_risk"),
        F.round(F.sum("line_amount"), 2).alias("total_net_amount"),
    )
)